In [5]:
from pathlib import Path
import shutil, subprocess, sys, torch

candidates = list(Path('/kaggle/input').rglob('scripts/finetune_lora.py'))
assert len(candidates) == 1, f'Expected one training script, found: {candidates}'
source = candidates[0].parent.parent
work = Path('/kaggle/working/timesfm-pjme-work')
if work.exists():
    shutil.rmtree(work)
shutil.copytree(source, work)
script = work / 'scripts/finetune_lora.py'
code = script.read_text()
old = 'dtype = torch.bfloat16 if device == "cuda" else torch.float32'
new = 'dtype = (torch.float16 if torch.cuda.get_device_capability(0)[0] < 8 else torch.bfloat16) if device == "cuda" else torch.float32'
assert old in code, 'Unexpected dtype line; refusing to patch the training script'
script.write_text(code.replace(old, new, 1))
print('GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0), 'CUDA devices:', torch.cuda.device_count(), flush=True)
print('Using FP16 on T4; all training windows are sampled from PJME train split (2,000 per epoch).', flush=True)
command = [sys.executable, '-u', str(script), '--data', str(work / 'data/pjme_hourly.csv'), '--model-id', 'google/timesfm-2.5-200m-transformers', '--context-len', '512', '--horizon-len', '24', '--epochs', '5', '--batch-size', '8', '--samples', '2000', '--patience', '0', '--output-dir', '/kaggle/working/timesfm-lora-pjme']
subprocess.run(command, cwd=work, check=True)

GPU: Tesla T4 capability: (7, 5) CUDA devices: 2
Using FP16 on T4; all training windows are sampled from PJME train split (2,000 per epoch).


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1162.22it/s]


trainable params: 1,382,912 || all params: 232,672,192 || trainable%: 0.5944
epoch=1/5 batch=25/250 loss=0.161935 seconds_per_batch=0.50
epoch=1/5 batch=50/250 loss=0.203089 seconds_per_batch=0.39
epoch=1/5 batch=75/250 loss=0.236259 seconds_per_batch=0.39
epoch=1/5 batch=100/250 loss=0.351895 seconds_per_batch=0.39
epoch=1/5 batch=125/250 loss=0.405725 seconds_per_batch=0.39
epoch=1/5 batch=150/250 loss=0.146907 seconds_per_batch=0.39
epoch=1/5 batch=175/250 loss=0.269489 seconds_per_batch=0.39
epoch=1/5 batch=200/250 loss=0.244308 seconds_per_batch=0.39
epoch=1/5 batch=225/250 loss=0.293121 seconds_per_batch=0.39
epoch=1/5 batch=250/250 loss=0.117623 seconds_per_batch=0.39
epoch=1/5 validation_loss=0.273837 epoch_seconds=105.1


epoch=2/5 batch=25/250 loss=0.169742 seconds_per_batch=0.39
epoch=2/5 batch=50/250 loss=0.194992 seconds_per_batch=0.39
epoch=2/5 batch=75/250 loss=0.196052 seconds_per_batch=0.39
epoch=2/5 batch=100/250 loss=0.111567 seconds_per_batch=0.39
epoch=2/5 batch=125/250 loss=0.173571 seconds_per_batch=0.38
epoch=2/5 batch=150/250 loss=0.331029 seconds_per_batch=0.39
epoch=2/5 batch=175/250 loss=0.239873 seconds_per_batch=0.39
epoch=2/5 batch=200/250 loss=0.153940 seconds_per_batch=0.40
epoch=2/5 batch=225/250 loss=0.208689 seconds_per_batch=0.39
epoch=2/5 batch=250/250 loss=0.322206 seconds_per_batch=0.39
epoch=2/5 validation_loss=0.273931 epoch_seconds=102.3
epoch=3/5 batch=25/250 loss=0.221342 seconds_per_batch=0.39
epoch=3/5 batch=50/250 loss=0.177037 seconds_per_batch=0.39
epoch=3/5 batch=75/250 loss=0.151094 seconds_per_batch=0.39
epoch=3/5 batch=100/250 loss=0.129249 seconds_per_batch=0.39
epoch=3/5 batch=125/250 loss=0.258220 seconds_per_batch=0.39
epoch=3/5 batch=150/250 loss=0.11228

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/working/timesfm-pjme-work/scripts/finetune_lora.py', '--data', '/kaggle/working/timesfm-pjme-work/data/pjme_hourly.csv', '--model-id', 'google/timesfm-2.5-200m-transformers', '--context-len', '512', '--horizon-len', '24', '--epochs', '5', '--batch-size', '8', '--samples', '2000', '--patience', '0', '--output-dir', '/kaggle/working/timesfm-lora-pjme'], returncode=0)

In [7]:
from pathlib import Path
import zipfile
root = Path('/kaggle/working')
adapter = root / 'timesfm-lora-pjme'
results = root / 'pjme-test-comparison'
assert (adapter / 'adapter_model.safetensors').exists(), 'Trained adapter not found'
assert (results / 'metrics.json').exists(), 'Full 908-window metrics not found'
out = root / 'pjme-timesfm-artifacts.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for base in (adapter, results):
        for path in base.rglob('*'):
            if path.is_file():
                z.write(path, path.relative_to(root))
print(out, out.stat().st_size, 'bytes')

/kaggle/working/pjme-timesfm-artifacts.zip 5735794 bytes


In [8]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/pjme-timesfm-artifacts.zip'))

/kaggle/working/pjme-timesfm-artifacts.zip

In [6]:
from pathlib import Path
import shutil, subprocess, sys, torch

# Locate the uploaded evaluation bundle and PJME hourly data.
compare_scripts = list(Path('/kaggle/input').rglob('scripts/compare_models.py'))
assert len(compare_scripts) == 1, f'Expected one compare_models.py, found: {compare_scripts}'
eval_source = compare_scripts[0].parent.parent
work = Path('/kaggle/working/timesfm-pjme-eval')
if work.exists():
    shutil.rmtree(work)
shutil.copytree(eval_source, work)
data_matches = list(Path('/kaggle/input').rglob('data/pjme_hourly.csv'))
assert len(data_matches) == 1, f'Expected one PJME CSV, found: {data_matches}'
data_path = data_matches[0]

# T4 is compute capability 7.5, so use FP16 for inference.
models_file = work / 'src/models.py'
model_code = models_file.read_text()
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
    model_code = model_code.replace('torch.bfloat16', 'torch.float16')
    models_file.write_text(model_code)
print('GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0), 'CUDA devices:', torch.cuda.device_count(), flush=True)
print('Data:', data_path, flush=True)
print('Running all test windows; inference batch size 32.', flush=True)
command = [sys.executable, '-u', str(work / 'scripts/compare_models.py'), '--data', str(data_path), '--adapter', '/kaggle/working/timesfm-lora-pjme', '--split', 'test', '--windows', '0', '--inference-batch-size', '32', '--output-dir', '/kaggle/working/pjme-test-comparison']
subprocess.run(command, cwd=work, check=True)
print('Outputs:', sorted(p.name for p in Path('/kaggle/working/pjme-test-comparison').glob('*')), flush=True)

GPU: Tesla T4 capability: (7, 5) CUDA devices: 2
Data: /kaggle/input/datasets/syedgufranahmed/timesfm-pjme-training-data/data/pjme_hourly.csv
Running all test windows; inference batch size 32.
{
  "model": "seasonal_naive",
  "evaluation_split": "test",
  "metrics": {
    "MAE": 3432.2824889867843,
    "RMSE": 4733.947982547805,
    "MASE": 1.596026275558017
  },
  "horizon_hours": 24,
  "context_length": 512,
  "windows_evaluated": 908,
  "hours_evaluated": 21792
}


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1212.00it/s]


{
  "model": "timesfm_zero_shot",
  "evaluation_split": "test",
  "metrics": {
    "MAE": 1582.7728467764834,
    "RMSE": 2214.4241645631078,
    "MASE": 0.7359962531641018
  },
  "horizon_hours": 24,
  "context_length": 512,
  "windows_evaluated": 908,
  "hours_evaluated": 21792
}


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1215.34it/s]


{
  "model": "timesfm_lora",
  "evaluation_split": "test",
  "metrics": {
    "MAE": 1536.445939432332,
    "RMSE": 2132.7119269382324,
    "MASE": 0.7144540398923631
  },
  "horizon_hours": 24,
  "context_length": 512,
  "windows_evaluated": 908,
  "hours_evaluated": 21792
}
Comparison saved to /kaggle/working/pjme-test-comparison
Outputs: ['comparison.png', 'lead_metrics.csv', 'metrics.json', 'predictions.csv', 'summary.csv']
